# Notebook 04 — Feature Engineering & Temporal Split

Tạo lag/rolling chỉ từ PM2.5 quan sát quá khứ, target `t+1` và `t+24`, sau đó chia theo năm: Train (2010–2012), Validation (2013), Test (2014). Các hàng target missing bị loại khỏi tập dùng cho mô hình; không nội suy target. Kiểm tra ranh giới thời gian để không giữ nhãn vượt qua split.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

# Tìm gốc repo có chứa file dữ liệu gốc để đường dẫn chạy được trên Colab.
DATA_RELATIVE = Path("PRSA_data_2010.1.1-2014.12.31.csv")
repo_candidates = [
    Path.cwd(),
    Path("/content/PPNCKH_5Bros"),
    Path("/content"),
]
PROJECT_ROOT = next(
    (root.resolve() for root in repo_candidates if (root / DATA_RELATIVE).exists()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu gốc: {DATA_RELATIVE}. "
        "Kiểm tra repo đã được clone vào /content/PPNCKH_5Bros và file raw nằm đúng vị trí."
    )

DATA_PATH = PROJECT_ROOT / DATA_RELATIVE
OUTPUT_ROOT = PROJECT_ROOT / "PPNCKH_5Bros/notebooks/phuong_nb/output"
DATA_OUT = OUTPUT_ROOT / "data"
FIG_OUT = OUTPUT_ROOT / "figures"
TABLE_OUT = OUTPUT_ROOT / "tables"
for folder in (DATA_OUT, FIG_OUT, TABLE_OUT):
    folder.mkdir(parents=True, exist_ok=True)

def load_raw_data():
    data = pd.read_csv(DATA_PATH)
    required = {"year", "month", "day", "hour"}
    missing_cols = required - set(data.columns)
    if missing_cols:
        raise ValueError(f"Thiếu cột thời gian: {sorted(missing_cols)}")
    data["datetime"] = pd.to_datetime(data[["year", "month", "day", "hour"]], errors="coerce")
    return data

print("Repo root:", PROJECT_ROOT)
print("Data file:", DATA_PATH)
print("Output root:", OUTPUT_ROOT)

Repo root: /content
Data file: /content/PRSA_data_2010.1.1-2014.12.31.csv
Output root: /content/PPNCKH_5Bros/notebooks/phuong_nb/output


In [2]:
df = load_raw_data()
df = df.loc[df["datetime"].notna()].sort_values("datetime").set_index("datetime")
df.index.name = "datetime"

if "pm2.5" not in df.columns:
    raise ValueError("Không tìm thấy cột pm2.5.")

# Không nội suy PM2.5. Lag/rolling chỉ sử dụng giá trị quan sát trước đó.
features = df.copy()
for lag in [1, 2, 3, 6, 12, 24, 48, 72]:
    features[f"pm2.5_lag_{lag}h"] = features["pm2.5"].shift(lag)

for window in [3, 6, 12, 24]:
    features[f"pm2.5_rolling_mean_{window}h"] = (
        features["pm2.5"].shift(1).rolling(window=window, min_periods=window).mean()
    )
features["pm2.5_rolling_std_24h"] = (
    features["pm2.5"].shift(1).rolling(window=24, min_periods=24).std()
)
features["pm2.5_rolling_max_24h"] = (
    features["pm2.5"].shift(1).rolling(window=24, min_periods=24).max()
)

# Timestamp target cụ thể giúp kiểm tra target có vượt ranh giới split không.
features["target_t1"] = features["pm2.5"].shift(-1)
features["target_t1_datetime"] = pd.Series(features.index, index=features.index).shift(-1)
features["target_t24"] = features["pm2.5"].shift(-24)
features["target_t24_datetime"] = pd.Series(features.index, index=features.index).shift(-24)
features["year"] = features.index.year

# Chỉ giữ các hàng có cả hai target quan sát; không dropna trên toàn bộ cột vì
# biến khí tượng khác có thể thiếu và việc xử lý chúng nên nằm trong model pipeline.
model_df = features.loc[features["target_t1"].notna() & features["target_t24"].notna()].copy()

def split_by_year(data):
    train = data[data["year"].between(2010, 2012)].copy()
    val = data[data["year"] == 2013].copy()
    test = data[data["year"] == 2014].copy()
    return train, val, test

train, validation, test = split_by_year(model_df)

# Kiểm tra target timestamp nằm trong cùng khoảng thời gian với split nguồn.
def check_target_boundaries(name, part, start_year, end_year):
    if part.empty:
        print(f"{name}: không có dòng.")
        return
    for col in ["target_t1_datetime", "target_t24_datetime"]:
        target_years = part[col].dt.year
        bad = ~target_years.between(start_year, end_year)
        if bad.any():
            print(f"WARNING: {name} có {int(bad.sum())} target vượt ranh giới theo {col}.")
        else:
            print(f"OK: {name} — {col} không vượt ranh giới năm {start_year}-{end_year}.")

check_target_boundaries("Train", train, 2010, 2012)
check_target_boundaries("Validation", validation, 2013, 2013)
check_target_boundaries("Test", test, 2014, 2014)

# Bỏ timestamp phụ khỏi CSV cuối? Giữ lại để audit ranh giới và loại khỏi X khi train.
train.to_csv(DATA_OUT / "beijing_train.csv")
validation.to_csv(DATA_OUT / "beijing_validation.csv")
test.to_csv(DATA_OUT / "beijing_test.csv")
model_df.to_csv(DATA_OUT / "beijing_hourly_features_targets.csv")

# Dictionary có phân loại target và timestamp target để tránh đưa target vào X.
target_cols = {"target_t1", "target_t24", "target_t1_datetime", "target_t24_datetime"}
feature_dictionary = pd.DataFrame({
    "column": model_df.columns,
    "role": [
        "target_or_target_timestamp" if c in target_cols else
        "split_metadata" if c == "year" else
        "candidate_feature"
        for c in model_df.columns
    ],
    "dtype": [str(model_df[c].dtype) for c in model_df.columns],
})
feature_dictionary.to_csv(TABLE_OUT / "feature_dictionary.csv", index=False)

print("Rows — train:", len(train), "| validation:", len(validation), "| test:", len(test))
print("Data outputs:", DATA_OUT)
print("Feature dictionary:", TABLE_OUT / "feature_dictionary.csv")

OK: Test — target_t1_datetime không vượt ranh giới năm 2014-2014.
OK: Test — target_t24_datetime không vượt ranh giới năm 2014-2014.


Rows — train: 23595 | validation: 8597 | test: 8538
Data outputs: /content/PPNCKH_5Bros/notebooks/phuong_nb/output/data
Feature dictionary: /content/PPNCKH_5Bros/notebooks/phuong_nb/output/tables/feature_dictionary.csv


## Quy tắc khi huấn luyện mô hình

- Không đưa `target_t1`, `target_t24`, `target_t1_datetime`, `target_t24_datetime` hoặc `year` vào ma trận đặc trưng `X`.
- Tạo `X` từ danh sách cột feature được chọn rõ ràng; tạo `y` riêng từ một target.
- Split theo năm được tạo trước khi huấn luyện. Nếu đánh giá forecast rolling-origin, cần quy định rõ dự báo nhiều bước có được phép sử dụng quan sát thực tế mới nhất hay không.
- Với lag `t-1` tại đầu Validation/Test, nó có thể tham chiếu quan sát ở thời gian trước đó. Đây là hợp lệ cho dự báo vận hành nếu giá trị đó thực sự đã quan sát tại thời điểm dự báo. Target timestamp phải nằm trong cùng split; các dòng ở cuối mỗi split có target vượt ranh giới sẽ bị đánh dấu/loại khỏi split.

In [3]:
# Loại những dòng target vượt ranh giới năm của từng split.
def keep_targets_inside_year_range(part, start_year, end_year):
    t1y = part["target_t1_datetime"].dt.year
    t24y = part["target_t24_datetime"].dt.year
    mask = t1y.between(start_year, end_year) & t24y.between(start_year, end_year)
    return part.loc[mask].copy()

train = keep_targets_inside_year_range(train, 2010, 2012)
validation = keep_targets_inside_year_range(validation, 2013, 2013)
test = keep_targets_inside_year_range(test, 2014, 2014)

train.to_csv(DATA_OUT / "beijing_train.csv")
validation.to_csv(DATA_OUT / "beijing_validation.csv")
test.to_csv(DATA_OUT / "beijing_test.csv")

print("Sau khi loại target vượt ranh giới:")
print("Train:", len(train), "| Validation:", len(validation), "| Test:", len(test))

Sau khi loại target vượt ranh giới:
Train: 23571 | Validation: 8573 | Test: 8538
